# Allergen Model Evaluation (Thresholds on Validation, Held-Out Test)

**Purpose:** calibrate thresholds on the **validation** set, run the **single** held-out
**test** evaluation that becomes the final reported numbers, and compare the ML model
against the official OFF allergen tags as a real-world signal.

## Methodology

- The model was chosen in `08_allergen_training` (`models/model_selection.json`).
- Per-class thresholds are tuned on validation (`find_best_thresholds`), not on test.
- The test set is evaluated **once** with those fixed thresholds — no iteration.
- **ML vs official tags** is the honest external check: how well does the model agree
  with manufacturer-declared allergens?

## Setup & Imports

In [ ]:
import os
import json
import ast
import warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

from sklearn.metrics import f1_score
from scipy.special import expit

from utils.data_utils import get_data_directories, load_labeled_data
from utils.text_processing import (
    get_allergen_list,
    clean_html,
    parse_official_tags,
    has_explicit_allergen_statement,
    rule_match,
)
from utils.model_utils import (
    load_model_and_tokenizer,
    find_best_thresholds,
    apply_thresholds,
    predict_ml,
)
from utils.evaluation import (
    print_classification_report,
    print_per_class_metrics,
    error_analysis,
    evaluate_rule_vs_official,
)

print("Imports complete.")

## 1. Load Chosen Model + Canonical Splits

In [ ]:
dirs = get_data_directories()
split_dir = os.path.join(dirs["final"], "splits")

def load_split(name):
    split_path = os.path.join(split_dir, name)
    if not os.path.exists(split_path):
        raise FileNotFoundError(
            f"Missing split file {split_path}. Run Notebook 07 (training data preparation) first."
        )
    d = pd.read_csv(split_path)
    d["labels"] = d["labels"].apply(ast.literal_eval)
    if "text" not in d.columns:
        d["text"] = d["ingredients_cleaned"]
    return d

val_df = load_split("val.csv")
test_df = load_split("test.csv")
val_texts = val_df["text"].tolist()
val_labels = np.array(val_df["labels"].tolist())
test_texts = test_df["text"].tolist()
test_labels = np.array(test_df["labels"].tolist())
allergen_list = get_allergen_list()

with open(os.path.join(dirs["models"], "model_selection.json")) as f:
    selection = json.load(f)
print(f"Chosen model: {selection['winner']}")

final_model_dir = os.path.join(dirs["models"], "mobilebert_allergen_final")
if not os.path.exists(final_model_dir):
    raise FileNotFoundError(
        f"Model directory not found: {final_model_dir}. Run Notebook 08 (allergen training) first."
    )
model, tokenizer, device = load_model_and_tokenizer(final_model_dir)
max_length = selection.get("max_len", 209)
print(f"Loaded canonical model (winner={selection['winner']}), max_length={max_length}")

## 2. Prediction Helper (handles ensemble winner)

In [ ]:
def predict_probs(texts, batch_size=8):
    if selection["winner"] == "ensemble":
        probs = []
        for seed in selection["models"]["ensemble_seeds"]:
            path = os.path.join(dirs["models"], "mobilebert_allergen_ensemble", f"seed_{seed}")
            m, tk, dev = load_model_and_tokenizer(path)
            _, p = predict_ml(texts, m, tk, dev, max_length=max_length, batch_size=batch_size)
            probs.append(p)
        return np.mean(probs, axis=0)
    _, probs = predict_ml(texts, model, tokenizer, device, max_length=max_length, batch_size=batch_size)
    return probs

## 3. Threshold Calibration on VALIDATION

In [ ]:
val_probs = predict_probs(val_texts)

# Recall-first thresholds (prioritize recall over precision): F-beta with beta=2 (recall
# weighted 4x vs precision) subject to a recall floor of 0.9. A missed allergen is the
# dangerous error, so thresholds should trade precision for recall on the rare classes.
RECALL_FIRST = True
if RECALL_FIRST:
    best_thresholds = find_best_thresholds(val_probs, val_labels, beta=2.0, min_recall=0.9)
else:
    best_thresholds = find_best_thresholds(val_probs, val_labels, beta=1.0, min_recall=None)

# Report the balanced-F1 thresholds alongside for the precision cost of recall-first
balanced_thresholds = find_best_thresholds(val_probs, val_labels, beta=1.0, min_recall=None)
print("Per-class thresholds (tuned on validation only):")
print(f"  {'allergen':<12} {'balanced-F1':>11} {'recall-first':>13}")
for i, a in enumerate(allergen_list):
    print(f"  {a:<12} {balanced_thresholds[i]:>11.2f} {best_thresholds[i]:>13.2f}")


def best_global_samples(probs, labels, step=0.01):
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.01, 0.99, step):
        f1 = f1_score(labels, (probs >= t).astype(int), average="samples", zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


best_t_global, best_samples_val = best_global_samples(val_probs, val_labels)
print(f"Best global threshold (samples F1): {best_t_global:.2f}  (val samples F1 {best_samples_val:.4f})")

# Persist the single source of truth for downstream notebooks + deployment
np.save(os.path.join(dirs["models"], "best_thresholds.npy"), best_thresholds)
hybrid_config = {
    "ml_thresholds": best_thresholds.tolist(),
    "rule_conf_threshold": 0.5,
    "mode": "hard_override",
    "winner": selection["winner"],
    "max_length": max_length,
    "threshold_objective": "recall_first",
    "threshold_beta": 2.0,
    "threshold_min_recall": 0.9,
}
with open(os.path.join(dirs["models"], "hybrid_config.json"), "w") as f:
    json.dump(hybrid_config, f, indent=2)
print("\nSaved best_thresholds.npy + hybrid_config.json (recall-first objective)")

## 4. Held-Out Test Evaluation (single, final)

Thresholds were fixed on validation; the test set is evaluated once. These are the
numbers to report.

In [ ]:
test_probs = predict_probs(test_texts)
test_preds = apply_thresholds(test_probs, best_thresholds)

print("=" * 64)
print("  FINAL HELD-OUT TEST EVALUATION")
print("  (thresholds tuned on validation only; test used once)")
print("=" * 64)
print_classification_report(test_labels, test_preds, allergen_list, prefix="Test Set (optimized thresholds)")
print_per_class_metrics(test_labels, test_preds, allergen_list, prefix="Per-class on test:")
print(f"\nSamples F1: {f1_score(test_labels, test_preds, average='samples', zero_division=0):.4f}")

error_analysis(test_texts, test_labels, test_preds, allergen_list, max_errors=3)

## 5. Non-Circular Ground-Truth Check: Hand-Labeled Hard Cases

Every F1 above compares predictions against labels that were **generated by the same rule
engine** we are evaluating — a form of circularity. This section breaks that loop:
`data/final/hard_cases.csv` holds **75 hard-case products** whose allergens were labeled
by a human (not the rule engine). They are drawn from the held-out test split and biased
toward the "difficult edge cases" the project cares about: facility cross-contamination
warnings, ambiguous Filipino terms, nut-free "nut" products (coconut, shea, water
chestnut, green tea), and chocolate/confectionery.

Labels follow the same **recall-first semantics** as the rest of the pipeline: a
"may contain X" cross-contamination warning counts as a positive (a risk is still a risk).

This is the non-circular reference: the only signal not derived from the rule engine being evaluated.

In [ ]:
# Non-circular ground truth: 75 hand-labeled hard cases (human labels, not rule-engine generated).
# Human (not rule-engine) labels, recall-first semantics (may-contain = positive).
HC_PATH = os.path.join(dirs["final"], "hard_cases.csv")
hc = pd.read_csv(HC_PATH)
hc_texts = hc["text"].fillna("").tolist()
hc_labels = hc[allergen_list].to_numpy()  # (75, 8) — matches rule_preds_hc / ml_preds_hc

# Rule engine
rule_preds_hc = np.array([[1 if rule_match(t, a) else 0 for a in allergen_list] for t in hc_texts])
# ML with recall-first thresholds (fixed on validation, never on this set)
hc_probs = predict_probs(hc_texts, batch_size=8)
ml_preds_hc = apply_thresholds(hc_probs, best_thresholds)
# Hybrid = recall-first union (rule_priority for every allergen)
hybrid_preds_hc = np.maximum(rule_preds_hc, ml_preds_hc)


def prf(t, p):
    tp = int(((p == 1) & (t == 1)).sum())
    fp = int(((p == 1) & (t == 0)).sum())
    fn = int(((p == 0) & (t == 1)).sum())
    pr = tp / max(tp + fp, 1)
    re = tp / max(tp + fn, 1)
    return pr, re, (2 * pr * re / (pr + re) if (pr + re) > 0 else 0.0)


print(f"HARD-CASE EVAL vs HUMAN LABELS (N={len(hc)})")
print(f"{'Allergen':<10} {'Rule P/R/F1':>25} {'ML P/R/F1':>25} {'Hybrid P/R/F1':>25}")
for j, a in enumerate(allergen_list):
    row = []
    for preds in (rule_preds_hc, ml_preds_hc, hybrid_preds_hc):
        pr, re, f1 = prf(hc_labels[:, j], preds[:, j])
        row.append(f"{pr:.3f}/{re:.3f}/{f1:.3f}")
    print(f"{a:<10} {row[0]:>25} {row[1]:>25} {row[2]:>25}")

print("\nSamples F1 vs human labels "
      f"(rule/ML/hybrid): {f1_score(hc_labels, rule_preds_hc, average='samples', zero_division=0):.3f} / "
      f"{f1_score(hc_labels, ml_preds_hc, average='samples', zero_division=0):.3f} / "
      f"{f1_score(hc_labels, hybrid_preds_hc, average='samples', zero_division=0):.3f}")

j = allergen_list.index("tree_nuts")
print("\ntree_nuts FP/FN breakdown:")
for name, preds in [("Rule", rule_preds_hc), ("ML", ml_preds_hc), ("Hybrid", hybrid_preds_hc)]:
    pr, re, f1 = prf(hc_labels[:, j], preds[:, j])
    fp = int(((hc_labels[:, j] == 0) & (preds[:, j] == 1)).sum())
    fn = int(((hc_labels[:, j] == 1) & (preds[:, j] == 0)).sum())
    print(f"  tree_nuts {name:6s}: {fp} FP, {fn} FN  P={pr:.3f} R={re:.3f}")

## 6. ML vs Official OFF Tags (real-world signal)

In [ ]:
# External check runs on the held-out test split only.
# Earlier versions scored the ML model against OFF tags on ALL 1057 products, which
# includes the 737 products the model was trained on (in-sample → inflated agreement).
# Restricting to the 158 held-out test products provides a non-circular signal:
# neither the rule engine nor the training labels are the ground truth here — the
# manufacturer-declared `allergens_tags` from Open Food Facts are.
CLEANED_PATH = os.path.join(dirs["processed"], "cleaned_dataset.csv")
df_off = test_df.copy()  # already holds code + text
off_tags = load_labeled_data(CLEANED_PATH)[["code", "allergens_tags"]]
df_off = df_off.merge(off_tags, on="code", how="left")

ml_probs_test = predict_probs(df_off["text"].tolist(), batch_size=8)
ml_preds_test = apply_thresholds(ml_probs_test, best_thresholds)
df_off["detected_allergens"] = [
    [allergen_list[i] for i, v in enumerate(row) if v == 1] for row in ml_preds_test
]
df_off["official_allergens_mapped"] = df_off["allergens_tags"].apply(
    lambda x: parse_official_tags(x) if pd.notna(x) and str(x) not in ("[]", "nan", "") else []
)

full_results = evaluate_rule_vs_official(df_off, allergen_list, prefix="ML vs OFF tags (held-out test only)")

df_off["explicit"] = df_off["ingredients_text_en"].apply(has_explicit_allergen_statement)
explicit_results = evaluate_rule_vs_official(
    df_off[df_off["explicit"]], allergen_list, prefix="ML vs OFF tags (explicit statements, held-out)"
)
print(f"\nProducts with explicit statements: {int(df_off['explicit'].sum())}")

## 7. Seed Variance (if the ensemble won)

In [ ]:
if selection["winner"] == "ensemble":
    print("Per-seed test samples F1 (ensemble averaging should beat most members):")
    for seed in selection["models"]["ensemble_seeds"]:
        path = os.path.join(dirs["models"], "mobilebert_allergen_ensemble", f"seed_{seed}")
        m, tk, dev = load_model_and_tokenizer(path)
        _, p = predict_ml(test_texts, m, tk, dev, max_length=max_length, batch_size=8)
        preds = apply_thresholds(p, best_thresholds)
        print(f"  Seed {seed:4d}: samples F1 {f1_score(test_labels, preds, average='samples', zero_division=0):.4f}")
    print(f"  Ensemble : samples F1 {f1_score(test_labels, test_preds, average='samples', zero_division=0):.4f}")
else:
    print(f"Single-model winner ({selection['winner']}) — per-seed spread not applicable.")

## Calibration note

`utils.model_utils.TemperatureScaling` is available if you want post-hoc probability
calibration: learn a temperature on the validation logits and re-scale before thresholding.
It was left out of the main flow to keep the reported numbers on a single, well-defined
thresholding pipeline.

**Next:** run `10_hybrid_evaluation.ipynb`.